# 0. Orientation: from one procedure call to a durable pipeline

You already know how Provium executes a procedure over artifacts. `provium-pipeline` adds the orchestration facts needed when that call becomes part of a repeatable, inspectable workflow.


## The object chain

Think in this order: **definition → compiled pipeline → input snapshot → run → tasks → dispatch → attempts → outputs**. A definition is intent. Compilation resolves catalogs and validates the graph. A run freezes that meaning plus its inputs. Tasks are the per-record/per-node work. A dispatch selects tasks to work now. Attempts own leases while workers execute. Outputs point to managed artifacts.

This separation is the main source of the library's size—and its recoverability. Each object answers a different question instead of overloading one mutable job record.


In [ ]:
import provium_pipeline as pipeline

public_concepts = {
    name: getattr(pipeline, name)
    for name in (
        'PipelineIdentifier', 'InputSet', 'PipelineRun', 'PipelineTask',
        'RunState', 'TaskState', 'DispatchId', 'AttemptId',
    )
}
assert set(public_concepts) == {
    'PipelineIdentifier', 'InputSet', 'PipelineRun', 'PipelineTask',
    'RunState', 'TaskState', 'DispatchId', 'AttemptId',
}
public_concepts


## Semantic state versus operational state

Semantic state says *what* should happen: graph, resolved procedures, configurations, input bindings, and output expectations. Operational state says *what happened*: run/task/dispatch states, attempts, leases, retry timing, and published outputs. Keeping them separate makes retries and recovery unable to silently change the computation.


In [ ]:
from provium_pipeline import RunState, TaskState

terminal_runs = {RunState.SUCCEEDED, RunState.FAILED, RunState.CANCELLED}
terminal_tasks = {
    TaskState.SUCCEEDED, TaskState.REUSED, TaskState.FAILED, TaskState.CANCELLED,
}
assert len(terminal_runs) == 3 and len(terminal_tasks) == 4
sorted(state.value for state in terminal_runs)


## What to notice

- A run is not a process; it is a durable, immutable computation request plus evolving status.
- A dispatch is not a run; it is one selection of a run's tasks.
- An attempt is not a task; it is one leased try at executing a task.
- Artifacts remain standard Provium artifacts. Pipeline adds storage locations, indexes, publication, and retention around them.

Next: [definitions and compilation](01-definitions-and-compilation.ipynb). Reference: [concepts](../docs/concepts.md) and [architecture](../docs/architecture.md).
